# Analysis 1 — Classification and comparison of medication-error groups among J01 (antibacterials for systemic use) reports

**Study:** Sandes et al. — Antibiotic dosing-related errors in Brazil (VigiMed).

**Purpose.** This notebook classifies J01 medication–event pairs from the Brazilian
spontaneous reporting database (VigiMed) into three mutually exclusive analytic
groups and compares them:
- **G1** — adverse events without medication error (reference)
- **G2** — medication error *without* a dosing component
- **G3** — dosing-related medication error

In [ ]:
import pandas as pd
import numpy as np

# ----------------------------------
# CONFIGURAÇÃO
# ----------------------------------
path_j01 = "filtered_j01_suspeito_nao_adm_y_analise_1225.csv"
pt_col = "PT"

# PTs que definem ERRO DE DOSE
pts_erro_dose = [
    "Administração de dose não ajustada",
    "Confusão com a dose do produto",
    "Confusão quanto ao regime do produto",
    "Dosagem incorreta administrada",
    "Dosagem não ajustada",
    "Dose adicional administrada",
    "Dose aumentada administrada",
    "Dose de reforço perdida",
    "Dose incorreta",
    "Dose incorreta administrada",
    "Dose incorreta administrada pelo dispositivo",
    "Dose incorreta administrada pelo produto",
    "Dose subterapêutica acidental",
    "Duração incorreta de administração do produto",
    "Erro de cálculo da dose",
    "Erro de titulação de medicamento",
    "Intoxicação acidental",
    "Omissão de dose do medicamento pelo dispositivo",
    "Omissão de dose do produto por erro",
    "Posologia inadequada de administração de produto",
    "Regime posológico incorreto",
    "Superdosagem acidental",
    "Taxa incorreta",
    "Taxa incorreta de administração do medicamento",
    "Titulação de dose do medicamento não realizada",
    "Dose subterapêutica",
    "Dose subterapêutica de radiação",
    "Dose subterapêutica prescrita",
    "Exposição a doses radioativas excessivas",
    "Problema relacionado à omissão de dose do produto",
    "Superdosagem",
    "Superdosagem por prescrição médica"
]



# PTs que definem ERRO DE MEDICAÇÃO (exceto dose)
pts_erro_med = [
    "Administração de formulação incorreta de produto",
    "Administração paravenosa de medicamento",
    "Análise de monitoramento terapêutico de medicamento não realizada",
    "Análise de monitoramento terapêutico de medicamento realizada incorretamente",
    "Ciclo de vacinação incompleto",
    "Circunstância ou informação capaz de resultar em erro de medicação",
    "Circunstância ou informação capaz de resultar em erro de utilização de dispositivo",
    "Componente único de um produto com dois componentes administrado",
    "Confusão com a embalagem do produto",
    "Confusão com o nome do produto",
    "Confusão com o produto",
    "Confusão de rótulo de produto",
    "Confusão na forma posológica do produto",
    "Confusão no uso do dispositivo",
    "Confusão pela aparência do produto",
    "Confusão relacionada ao desenho do produto",
    "Cronograma de descontinuação do produto inadequado",
    "Descarte incorreto de produto",
    "Dispositivo contraindicado utilizado",
    "Dispositivo errado",
    "Dispositivo errado selecionado",
    "Dispositivo incorreto utilizado",
    "Dispositivo vencido utilizado",
    "Erro de administração de produto interceptado",
    "Erro de armazenamento de produto",
    "Erro de armazenamento de produto interceptado",
    "Erro de dispensação de produto interceptado",
    "Erro de dispensação do dispositivo",
    "Erro de dispensação do produto",
    "Erro de implantação de dispositivo",
    "Erro de medicação",
    "Erro de medicação de interação entre medicamentos indicada no documento de referência",
    "Erro de medicação de interação indicada no documento de referência do medicamento com álcool",
    "Erro de medicação de interação indicada no documento de referência entre o medicamento e a doença",
    "Erro de medicação devido à interação medicamento e genética descrito no documento de referência",
    "Erro de medicação devido à interação medicamento-alimento descrita em bula",
    "Erro de medicação interceptado",
    "Erro de medicação na transferência assistencial",
    "Erro de monitoramento de dispositivo médico",
    "Erro de monitoramento de produto interceptado",
    "Erro de monitoramento do produto",
    "Erro de preparação de produto interceptado",
    "Erro de preparação do produto",
    "Erro de prescrição de produto interceptado",
    "Erro de prescrição do produto",
    "Erro de programação do dispositivo",
    "Erro de retirada de prescrição",
    "Erro de seleção de produto interceptado",
    "Erro de seleção do produto",
    "Erro de substituição do produto",
    "Erro de terapia duplicada",
    "Erro de transcrição de medicação",
    "Erro de utilização do dispositivo",
    "Erro de vacinação",
    "Erros de administração de produto",
    "Exposição acidental a embalagem de produto",
    "Exposição acidental a embalagem de produto por uma criança",
    "Exposição acidental a produto",
    "Exposição acidental a produto por pessoa idosa",
    "Exposição acidental a produto por uma criança",
    "Exposição acidental da criança ao produto interceptada",
    "Exposição por contato com a pele",
    "Exposição por contato com o olho",
    "Exposição por contato direto",
    "Falha da tampa do produto resistente a crianças",
    "Falha de suspensão de medicamento",
    "Falta de rotação do local de administração",
    "Falta de rotação do local de aplicação",
    "Falta de rotação do local de infusão",
    "Falta de rotação do local de injeção",
    "Falta de rotação do local de vacinação",
    "Hipersensibilidade documentada a produto administrado",
    "Ingestão acidental de dispositivo",
    "Ingestão acidental de dispositivo por uma criança",
    "Medicamento administrado em dispositivo incorreto",
    "Medicamento dispensado ao paciente incorreto",
    "Medicamento incorreto",
    "Obtenção de produto incorreto",
    "Paciente errado",
    "Paciente incorreto recebeu o produto",
    "Paciente incorreto selecionado interceptado",
    "Problema relacionado ao código de barras do produto",
    "Procedimento de monitoramento de dispositivo não realizado",
    "Procedimento de monitoramento de medicamento não realizado",
    "Procedimento de monitoramento de medicamento realizado incorretamente",
    "Produto administrado a paciente de idade inadequada",
    "Produto administrado em contexto inapropriado",
    "Produto administrado em local inadequado",
    "Produto administrado por pessoa errada",
    "Produto contraindicado administrado",
    "Produto contraindicado prescrito",
    "Produto descontinuado administrado",
    "Produto incorreto administrado",
    "Produto incorreto armazenado",
    "Produto recolhido do mercado administrado",
    "Produto vencido administrado",
    "Técnica asséptica inadequada na utilização do produto",
    "Técnica incorreta no processo de utilização de dispositivo",
    "Técnica incorreta no processo de utilização de produto",
    "Transfusão com sangue incompatível",
    "Uso acidental de placebo",
    "Uso de abreviação propensa a erros",
    "Uso múltiplo de produto de uso único",
    "Uso não intencional para indicação não aprovada",
    "Via errada",
    "Via incorreta de administração do produto",
    "Administração de produto interrompida",
    "Complicação da extração do sistema de administração do dispositivo",
    "Complicação da inserção de dispositivo",
    "Complicação da remoção de dispositivo",
    "Complicação de implantação",
    "Contraindicação à vacinação",
    "Contraindicação ao tratamento médico",
    "Déficit de conhecimento sobre o produto",
    "Dificuldade para utilizar o dispositivo",
    "Dispositivo de baixa qualidade utilizado",
    "Embalagem do produto difícil de abrir",
    "Espessamento de prótese valvular",
    "Exposição a dispositivo contaminado",
    "Exposição ocupacional a produto",
    "Exposição ocupacional a radiação",
    "Exposição por dispositivo contaminado",
    "Exposição por inalação",
    "Exposição por ingestão",
    "Exposição por meio do(a) parceiro(a)",
    "Exposição por via desconhecida",
    "Extração não intencional de dispositivo médico",
    "Falha de condição adicional para uso sem prescrição médica",
    "Incompatibilidade entre dispositivos",
    "Intercâmbio de produtos de vacina",
    "Lesão associada a dispositivo",
    "Mau funcionamento de válvula cardíaca protética",
    "Mau funcionamento do dispositivo",
    "Mau funcionamento do sistema de administração de medicamento",
    "Medicamento eficaz para indicação não aprovada",
    "Medicamento ineficaz para indicação não aprovada",
    "Mistura de produtos",
    "Não aderência ao tratamento",
    "Problema com condição adicional para uso sem prescrição médica",
    "Problema com o dessecante do produto",
    "Problema de atributo de segurança do dispositivo",
    "Problema de comunicação do produto",
    "Problema de conexão do dispositivo",
    "Problema de desvio de temperatura do produto",
    "Problema de dispensação do produto",
    "Problema de infusão do dispositivo",
    "Problema de interação entre medicamento e genética descrito no documento de referência",
    "Problema de interação medicamentosa com álcool descrita no documento de referência",
    "Problema de interação medicamentosa com alimento descrita no documento de referência",
    "Problema de interação medicamentosa descrita no documento de referência",
    "Problema de manutenção do dispositivo",
    "Problema de medicamentosa com doença descrita no documento de referência",
    "Problema de posicionamento do dispositivo",
    "Problema de preparação do produto",
    "Problema de prescrição do produto",
    "Problema de qualidade na composição do produto",
    "Problema de qualidade na reconstituição do produto",
    "Problema de utilização de dispositivo",
    "Problema de utilização de produto",
    "Problema na agulha",
    "Problema na data de validade do produto",
    "Problema na seringa",
    "Problema no mecanismo de liberação do produto",
    "Problema no número de identificação do produto",
    "Problema no número do lote do produto",
    "Problema no sistema de administração do medicamento",
    "Problema relacionado à aderência do dispositivo",
    "Problema relacionado à adesão do produto",
    "Problema relacionado à embalagem do produto",
    "Problema relacionado à utilização de dispositivo",
    "Problema relacionado ao rótulo do produto",
    "Produto administrado por provedor não autorizado",
    "Produto com problema de qualidade administrado",
    "Produto de baixa qualidade administrado",
    "Produto dispensado por fornecedor não autorizado",
    "Produto prescrito por fornecedor não autorizado",
    "Produto terapêutico eficaz para indicação não aprovada",
    "Produto terapêutico ineficaz para indicação não aprovada",
    "Queixa sobre a utilização do produto",
    "Queixa sobre o conteúdo das informações do produto",
    "Rótulo do produto no produto errado",
    "Uso de produto em ambiente terapêutico não aprovado",
    "Uso de produto em indicação não aprovada",
    "Confusão quanto à potência do produto",
    "Potência incorreta",
    "Forma posológica incorreta",
    "Forma posológica incorreta de produto administrada",
    "Formulação de dose incorreta",
    "Problema da forma de dosagem do produto"
]

# ----------------------------------
# LEITURA
# ----------------------------------
df = pd.read_csv(path_j01, sep=",", decimal=".", encoding="utf-8")
df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

if pt_col not in df.columns:
    raise ValueError(f"Coluna '{pt_col}' não encontrada.")

# ----------------------------------
# NORMALIZAÇÃO PT
# ----------------------------------
df[pt_col] = (
    df[pt_col]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["_PT_NORM"] = df[pt_col].str.upper()

pts_dose_norm = {p.strip().upper() for p in pts_erro_dose}
pts_med_norm  = {p.strip().upper() for p in pts_erro_med}

# ----------------------------------
# FLAGS
# ----------------------------------
is_dose = df["_PT_NORM"].isin(pts_dose_norm)
is_med  = df["_PT_NORM"].isin(pts_med_norm)

# ----------------------------------
# GRUPO ANALÍTICO (hierárquico)
# ----------------------------------
df["GRUPO_ANALITICO"] = np.select(
    [is_dose, (~is_dose) & is_med],
    ["G3_ERRO_DOSE", "G2_ERRO_MED_SEM_DOSE"],
    default="G1_SEM_ERRO"
)

# ----------------------------------
# CHECAGENS
# ----------------------------------
print("\nDistribuição GRUPO_ANALITICO:")
print(df["GRUPO_ANALITICO"].value_counts(dropna=False))

# Sanidade: nenhum PT de dose fora de G3
erro = df.loc[is_dose & (df["GRUPO_ANALITICO"] != "G3_ERRO_DOSE")]
print("Inconsistências (esperado 0):", len(erro))

df_base = df.drop(columns=["_PT_NORM"], errors="ignore")

Tabela 1 (descritiva)

Perfil por SEXO, FAIXA_ETARIA, UF, TIPO_ENTRADA_VIGIMED, GRAVE_y

Frequências e percentuais

Sem p-valor

In [ ]:
# =========================
# CONFIG
# =========================
df = df_base.copy()

COL_GROUP = "GRUPO_ANALITICO"
COL_SEX = "SEXO"
COL_AGE = "FAIXA_ETARIA"
COL_ENTRY = "TIPO_ENTRADA_VIGIMED"
COL_SEVERITY = "GRAVE_y"   # ajuste se necessário

# =========================
# 1) LOAD
# =========================
df = df_base.copy()
df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

# =========================
# 2) GROUPS (EN + ORDER)
# =========================
group_order = ["G1_SEM_ERRO", "G2_ERRO_MED_SEM_DOSE", "G3_ERRO_DOSE"]
group_labels = {"G1_SEM_ERRO": "Group 1", "G2_ERRO_MED_SEM_DOSE": "Group 2", "G3_ERRO_DOSE": "Group 3"}

df[COL_GROUP] = df[COL_GROUP].astype(str).str.strip()
df = df[df[COL_GROUP].isin(group_order)].copy()

df["GROUP_EN"] = df[COL_GROUP].map(group_labels)
df["GROUP_EN"] = pd.Categorical(df["GROUP_EN"], categories=[group_labels[g] for g in group_order], ordered=True)

# =========================
# 3) SEX (EN)
# =========================
df[COL_SEX] = df[COL_SEX].fillna("").astype(str).str.strip().str.upper()

def map_sex(x):
    if "FEM" in x:
        return "Female"
    if "MASC" in x:
        return "Male"
    return "Not reported / Unknown"

df["SEX_EN"] = df[COL_SEX].apply(map_sex)
df["SEX_EN"] = pd.Categorical(df["SEX_EN"], categories=["Female", "Male", "Not reported / Unknown"], ordered=True)

# =========================
# 4) AGE GROUP (EN + REQUIRED ORDER)
# =========================
df[COL_AGE] = df[COL_AGE].fillna("").astype(str).str.strip()

age_map = {
    "Neonato (0-30 dias)": "Neonate (0-30 days)",
    "Infantil (31 dias - 5 anos)": "Infant (31 days - 5 years)",
    "Criança (6-12 anos)": "Child (6-12 years)",
    "Adolescente (13-18 anos)": "Adolescent (13-18 years)",
    "Adulto (19-64 anos)": "Adult (19-64 years)",
    "Idoso (65+ anos)": "Elderly (65+ years)",
    "Ignorado": "Not reported / Unknown",
    "": "Not reported / Unknown",
}

df["AGE_EN"] = df[COL_AGE].map(age_map).fillna("Not reported / Unknown")

age_order = [
    "Neonate (0-30 days)",
    "Infant (31 days - 5 years)",
    "Child (6-12 years)",
    "Adolescent (13-18 years)",
    "Adult (19-64 years)",
    "Elderly (65+ years)",
    "Not reported / Unknown",
]
df["AGE_EN"] = pd.Categorical(df["AGE_EN"], categories=age_order, ordered=True)


def apply_entry_vigimed_mapping(
    df: pd.DataFrame,
    col_entry: str,
    out_col: str = "ENTRY_EN",
):
    """
    Replica o bloco 5 da Tabela 1:
    - limpeza NBSP + trim
    - mapeamento por dicionário
    - fillna -> Unknown
    - replace unification_fix
    - Categorical com ordem estável (filtrando categorias existentes)
    """
    # 1) Limpeza profunda
    def clean_text(x):
        if pd.isna(x) or x == "":
            return ""
        x = str(x)
        x = x.replace("\xa0", " ").replace("\u00A0", " ")
        x = " ".join(x.split())
        return x

    df[col_entry] = df[col_entry].apply(clean_text)

    # 2) Dicionário de tradução (colapsando indústria -> Pharmaceutical companies)
    entry_map = {
        "Empresas Farmacêuticas": "Pharmaceutical companies",
        "Pacientes e Profissionais de Saúde": "Patients and healthcare professionals",
        "Serviços de Saúde": "Healthcare services",
        "Serviços de Vacinação": "Vaccination services",
        "VigiFlow eForms": "VigiFlow eForms",
        "VigiMobile": "VigiMobile",
        "eReporting - Indústria, Carga E2B": "Pharmaceutical companies",
        "eReporting - Indústria, Entrada manual de dados": "Pharmaceutical companies",
    }

    # 3) Mapeamento
    df[out_col] = df[col_entry].map(entry_map)

    # 4) Não mapeado -> Unknown
    df[out_col] = df[out_col].fillna("Not reported / Unknown")

    # Unificação extra (mantida igual ao seu código)
    unification_fix = {
        "Empresas Farmacêuticas": "Pharmaceutical companies",
        "Pacientes e Profissionais de Saúde": "Patients and healthcare professionals",
    }
    df[out_col] = df[out_col].replace(unification_fix)

    # 5) Ordem final
    entry_order = [
        "Healthcare services",
        "Pharmaceutical companies",
        "Patients and healthcare professionals",
        "VigiMobile",
        "VigiFlow eForms",
        "Vaccination services",
        "Not reported / Unknown",
    ]

    existing_cats = set(df[out_col].unique())
    final_cats = [x for x in entry_order if x in existing_cats] + [
        x for x in existing_cats if x not in entry_order
    ]

    df[out_col] = pd.Categorical(df[out_col], categories=final_cats, ordered=True)

    return df


# =========================
# 6) SEVERITY (EN)
# =========================
if COL_SEVERITY not in df.columns:
    raise ValueError(f"Column '{COL_SEVERITY}' not found. Update COL_SEVERITY to match your dataset.")

df[COL_SEVERITY] = df[COL_SEVERITY].fillna("").astype(str).str.strip().str.upper()

def map_sev(x):
    if x in ["SIM", "YES", "Y", "TRUE", "1"]:
        return "Yes"
    if x in ["NAO", "NÃO", "NO", "N", "FALSE", "0"]:
        return "No"
    return "Not reported / Unknown"

df["SEVERITY_EN"] = df[COL_SEVERITY].apply(map_sev)
df["SEVERITY_EN"] = pd.Categorical(df["SEVERITY_EN"], categories=["Yes", "No", "Not reported / Unknown"], ordered=True)

# =========================
# 7) TABLE BUILDER
# =========================
def table_block(df, var_col, var_label, group_col="GROUP_EN", order=None):
    tmp = df[[var_col, group_col]].copy()

    if order is not None:
        tmp[var_col] = pd.Categorical(tmp[var_col], categories=order, ordered=True)

    tab = pd.crosstab(tmp[var_col], tmp[group_col], dropna=False)

    group_cols = list(tmp[group_col].cat.categories)
    for gc in group_cols:
        if gc not in tab.columns:
            tab[gc] = 0
    tab = tab[group_cols]

    col_totals = tab.sum(axis=0)
    pct = tab.div(col_totals, axis=1) * 100

    rows = []
    for i, cat in enumerate(tab.index):
        row = {"Variable": var_label if i == 0 else "", "Category": str(cat)}
        for gc in group_cols:
            row[gc] = f"{int(tab.loc[cat, gc]):,} ({pct.loc[cat, gc]:.2f}%)"
        rows.append(row)
    return pd.DataFrame(rows)

# Group N headers
group_n = df.groupby("GROUP_EN").size().reindex(df["GROUP_EN"].cat.categories)
group_headers = [f"{g}\nN = {int(group_n[g]):,}" for g in df["GROUP_EN"].cat.categories]
rename_groups = {old: new for old, new in zip(df["GROUP_EN"].cat.categories, group_headers)}

#  ENTRY IN VIGIMED (EN)
df = apply_entry_vigimed_mapping(df, COL_ENTRY, out_col="ENTRY_EN")

# precisa existir aqui porque você usa no table_block(order=entry_order)
entry_order = [
    "Healthcare services",
    "Pharmaceutical companies",
    "Patients and healthcare professionals",
    "VigiMobile",
    "VigiFlow eForms",
    "Vaccination services",
    "Not reported / Unknown",
]


blocks = []
blocks.append(table_block(df, "SEX_EN", "Sex", order=["Female", "Male", "Not reported / Unknown"]))
blocks.append(table_block(df, "AGE_EN", "Age Group", order=age_order))
blocks.append(table_block(df, "ENTRY_EN", "Entry in VigiMed", order=entry_order))
blocks.append(table_block(df, "SEVERITY_EN", "Severity", order=["Yes", "No", "Not reported / Unknown"]))

table1 = pd.concat(blocks, ignore_index=True).rename(columns=rename_groups)


# Preview
pd.set_option("display.max_rows", 120)
table1.head(60)


In [ ]:
from docx import Document
from docx.shared import Pt

# Criar documento Word
doc = Document()

# Título da tabela (formato de artigo)
title = doc.add_paragraph("Table 1. Characteristics of J01 medication–event pairs by error classification")
title.runs[0].bold = True

doc.add_paragraph("")  # linha em branco

# Criar tabela
table = doc.add_table(rows=1, cols=len(table1.columns))
table.style = "Table Grid"

# Cabeçalho
hdr_cells = table.rows[0].cells
for i, col in enumerate(table1.columns):
    hdr_cells[i].text = col

# Corpo da tabela
for _, row in table1.iterrows():
    row_cells = table.add_row().cells
    for i, val in enumerate(row):
        row_cells[i].text = str(val)

# Ajuste básico de fonte (opcional, mas recomendável)
for row in table.rows:
    for cell in row.cells:
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.font.size = Pt(9)

# Salvar
out_path = "Table1_J01_errors_y.docx"
doc.save(out_path)

print(f"Arquivo salvo: {out_path}")


Estatística de proporções: 

Análise Multivariada

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import unicodedata

# ==============================================================================
# CONFIGURAÇÕES E LEITURA
# ==============================================================================
df = df_base.copy()
PATH_OUT = "Table_MedicationError_vs_NoError.csv"

COL_GROUP = "GRUPO_ANALITICO"
COL_SEX = "SEXO"
COL_AGE = "FAIXA_ETARIA"
COL_ENTRY = "TIPO_ENTRADA_VIGIMED"
COL_CLUSTER = "IDENTIFICACAO_NOTIFICACAO"

if COL_CLUSTER not in df.columns:
    raise ValueError(f"Coluna '{COL_CLUSTER}' não encontrada. Necessária para o cálculo estatístico.")

required = [COL_GROUP, COL_SEX, COL_AGE, COL_ENTRY, "IDENTIFICACAO_NOTIFICACAO"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Colunas ausentes no CSV de entrada: {missing}")

# Limpeza básica de colunas
df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

# ==============================================================================
# 1. PREPARAÇÃO E LIMPEZA DOS DADOS (O "Gold Standard" que definimos)
# ==============================================================================

def normalize_text(x):
    """Limpeza robusta: Remove NBSP, acentos e põe em caixa alta."""
    if pd.isna(x) or x == "": return "NOT_REPORTED"
    x = str(x)
    # AQUI ESTÁ A CORREÇÃO MÁGICA:
    x = x.replace("\xa0", " ").replace("\u00A0", " ") 
    x = " ".join(x.split()) 
    x = unicodedata.normalize("NFKD", x)
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    return x.upper()

# --- FILTRO DOS GRUPOS ---
df[COL_GROUP] = df[COL_GROUP].fillna("").astype(str).str.strip()
valid_groups = ["G1_SEM_ERRO", "G2_ERRO_MED_SEM_DOSE", "G3_ERRO_DOSE"]
df = df[df[COL_GROUP].isin(valid_groups)].copy()


# --- SEXO ---
df["SEX_EN"] = df[COL_SEX].apply(lambda x: "Female" if "FEM" in normalize_text(x) else ("Male" if "MASC" in normalize_text(x) else "Not reported / Unknown"))
# Referência: Male
df["SEX_EN"] = pd.Categorical(df["SEX_EN"], categories=["Male", "Female", "Not reported / Unknown"], ordered=True)

# --- IDADE ---
age_map = {
    "Neonato (0-30 dias)": "Neonate (0-30 days)",
    "Infantil (31 dias - 5 anos)": "Infant (31 days - 5 years)",
    "Criança (6-12 anos)": "Child (6-12 years)",
    "Adolescente (13-18 anos)": "Adolescent (13-18 years)",
    "Adulto (19-64 anos)": "Adult (19-64 years)",
    "Idoso (65+ anos)": "Elderly (65+ years)",
}
df["AGE_EN"] = df[COL_AGE].str.strip().map(age_map).fillna("Not reported / Unknown")

# Ordem Visual (para a tabela final)
age_display_order = [
    "Neonate (0-30 days)", "Infant (31 days - 5 years)", "Child (6-12 years)",
    "Adolescent (13-18 years)", "Adult (19-64 years)", "Elderly (65+ years)", "Not reported / Unknown"
]
# Ordem do Modelo (Referência = Adulto)
age_model_order = ["Adult (19-64 years)"] + [x for x in age_display_order if x != "Adult (19-64 years)"]
df["AGE_EN"] = pd.Categorical(df["AGE_EN"], categories=age_model_order, ordered=True)


# =========================
#ENTRY IN VIGIMED (EN)
# =========================
df = apply_entry_vigimed_mapping(df, COL_ENTRY, out_col="ENTRY_EN")


def get_model_results(data, formula, outcome_label, cluster_col=COL_CLUSTER):
    """
    Roda GLM Poisson com erro-padrão clusterizado por notificação.
    """
    try:
        # Tenta rodar com Cluster (cov_type='cluster')
        fit = smf.glm(formula=formula, data=data, family=sm.families.Poisson()).fit(
            cov_type="cluster",
            cov_kwds={"groups": data[cluster_col]}
        )
    except Exception as e:
        print(f"Erro no cluster ({formula}): {e}. Tentando HC0 simples...")
        # Fallback se falhar o cluster (ex: apenas 1 caso)
        try:
             fit = smf.glm(formula=formula, data=data, family=sm.families.Poisson()).fit(cov_type="HC0")
        except:
             return pd.DataFrame() # Falha total

    params = fit.params
    conf = fit.conf_int()
    conf.columns = ["Lower", "Upper"]

    results = []
    for term in params.index:
        if term == "Intercept": continue

        # Lógica para extrair nome limpo da variável
        if "C(" in term:
            var_part = term.split("C(")[1].split(")")[0]
            cat_part = term.split("[T.")[1].rstrip("]")
        else:
            var_part = term
            cat_part = term

        pr = np.exp(params[term])
        lower = np.exp(conf.loc[term, "Lower"])
        upper = np.exp(conf.loc[term, "Upper"])

        results.append({
            "Variable": var_part,
            "Category": cat_part,
            outcome_label: f"{pr:.2f} ({lower:.2f}–{upper:.2f})"
        })
    return pd.DataFrame(results)


def run_any_error_analysis(df_full):
    """
    Outcome:
        1 = any medication error (G2 or G3)
        0 = no medication error (G1)

    Produces:
        - counts and percentages among error pairs
        - crude PRs
        - adjusted PRs
    """

    valid_groups = [
        "G1_SEM_ERRO",
        "G2_ERRO_MED_SEM_DOSE",
        "G3_ERRO_DOSE",
    ]

        # Select G1, G2, and G3
    df_sub = df_full[
        df_full[COL_GROUP].isin(valid_groups)
    ].copy()

    # Identify reporting-source categories with N < 10
    entry_counts = df_sub[
        "ENTRY_EN"
    ].value_counts(dropna=False)

    excluded_entries = entry_counts[
        entry_counts < 10
    ]

    print(
        "Reporting-source categories excluded "
        "from regression because N < 10:"
    )
    print(excluded_entries)

    # Retain reporting-source categories with N >= 10
    valid_entries = entry_counts[
        entry_counts >= 10
    ].index

    df_sub = df_sub[
        df_sub["ENTRY_EN"].isin(valid_entries)
    ].copy()

    # Outcome:
    # 1 = any medication error (G2 or G3)
    # 0 = no medication error (G1)
    df_sub["Y"] = df_sub[COL_GROUP].isin([
        "G2_ERRO_MED_SEM_DOSE",
        "G3_ERRO_DOSE",
    ]).astype(int)

    variables = [
        "SEX_EN",
        "AGE_EN",
        "ENTRY_EN",
    ]

    # Remove categories excluded from the analytical dataset
    for variable in variables:
        df_sub[variable] = (
            df_sub[variable]
            .cat.remove_unused_categories()
        )
    # --------------------------------------------------------
    # Counts among medication-error pairs
    # --------------------------------------------------------

    df_cases = df_sub[df_sub["Y"] == 1].copy()
    total_cases = len(df_cases)

    counts_list = []

    for variable in variables:

        categories = df_sub[variable].cat.categories

        counts = (
            df_cases[variable]
            .value_counts()
            .reindex(categories)
            .fillna(0)
        )

        percentages = counts / total_cases * 100

        for category in categories:
            counts_list.append({
                "Variable": variable,
                "Category": category,
                "n_cases_AnyError": int(counts[category]),
                "pct_cases_AnyError":
                    f"{percentages[category]:.1f}%",
            })

    df_counts = pd.DataFrame(counts_list)

    # --------------------------------------------------------
    # Crude PRs
    # --------------------------------------------------------

    crude_results = []

    for variable in variables:

        reference = df_sub[variable].cat.categories[0]

        crude_results.append(pd.DataFrame([{
            "Variable": variable,
            "Category": reference,
            "Crude_PR_AnyError": "1.00 (Reference)",
        }]))

        result = get_model_results(
            df_sub,
            f"Y ~ C({variable})",
            "Crude_PR_AnyError",
        )

        crude_results.append(result)

    df_crude = pd.concat(
        crude_results,
        ignore_index=True,
    )

    # --------------------------------------------------------
    # Adjusted model
    # --------------------------------------------------------

    formula_adjusted = (
        "Y ~ C(SEX_EN) + C(AGE_EN) + C(ENTRY_EN)"
    )

    adjusted_raw = get_model_results(
        df_sub,
        formula_adjusted,
        "Adj_PR_AnyError",
    )

    adjusted_references = []

    for variable in variables:

        reference = df_sub[variable].cat.categories[0]

        adjusted_references.append({
            "Variable": variable,
            "Category": reference,
            "Adj_PR_AnyError": "1.00 (Reference)",
        })

    df_adjusted = pd.concat(
        [
            pd.DataFrame(adjusted_references),
            adjusted_raw,
        ],
        ignore_index=True,
    )

    # --------------------------------------------------------
    # Final merge
    # --------------------------------------------------------

    result = pd.merge(
        df_counts,
        df_crude,
        on=["Variable", "Category"],
        how="left",
    )

    result = pd.merge(
        result,
        df_adjusted,
        on=["Variable", "Category"],
        how="left",
    )

    return result, df_sub
# ==============================================================================
# 3. EXECUÇÃO
# ==============================================================================

final_table, df_model = run_any_error_analysis(df)

print("Outcome distribution:")
print(df_model["Y"].value_counts().sort_index())

# ==============================================================================
# 4. FORMATAÇÃO FINAL (ORDEM VISUAL CORRETA)
# ==============================================================================

# Definir a ordem exata das linhas
var_order = ["SEX_EN", "AGE_EN", "ENTRY_EN"]
cat_sorter = {
    "SEX_EN": list(df["SEX_EN"].cat.categories),
    "AGE_EN": age_display_order, # Usa a ordem visual, não a do modelo
    "ENTRY_EN": list(df["ENTRY_EN"].cat.categories)
}

final_table["var_rank"] = final_table["Variable"].map(lambda x: var_order.index(x))
final_table["cat_rank"] = final_table.apply(lambda row: cat_sorter[row["Variable"]].index(row["Category"]) if row["Category"] in cat_sorter[row["Variable"]] else 99, axis=1)

final_table = final_table.sort_values(by=["var_rank", "cat_rank"])

# Limpeza estética
labels_map = {
    "SEX_EN": "Sex",
    "AGE_EN": "Age Group",
    "ENTRY_EN": "Reporter / Entry Type"
}
final_table["Variable"] = final_table["Variable"].map(labels_map)

# Colunas finais organizadas
cols_final = [
    "Variable",
    "Category",
    "n_cases_AnyError",
    "pct_cases_AnyError",
    "Crude_PR_AnyError",
    "Adj_PR_AnyError",
]

final_table = final_table[cols_final]
final_table.to_csv(PATH_OUT, index=False, sep=";", encoding="utf-8-sig")

print(f"✅ Tabela concluída! Salva em: {PATH_OUT}")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 1000)
print(final_table.head(20))

In [ ]:
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

# =========================================================
# 1) EXCLUSÕES ANTES DE SALVAR (ajuste conforme sua decisão)
# =========================================================
# Excluir "Vaccination services" da Tabela 2
final_table = final_table[final_table["Category"] != "Vaccination services"].copy()


# =========================================================
# 2) SALVAR EM WORD (DOCX)
# =========================================================
out_docx = "Table_Final_Crude_and_Adjusted_y_so1grupo.docx"

doc = Document()

# Título
title = doc.add_paragraph(
    "Crude and adjusted prevalence ratios for "
    "medication-error classification among J01 "
    "drug–event pairs"
)
title.runs[0].bold = True
title.alignment = WD_ALIGN_PARAGRAPH.LEFT

# Nota/rodapé (opcional; ajuste ao seu Methods)
note = doc.add_paragraph(
    "The outcome comprised any medication error, "
    "including dosing- and non-dosing-related errors. "
    "The reference outcome group comprised drug–event "
    "pairs without medication errors. Crude PRs were "
    "estimated using univariable models, and adjusted "
    "PRs using a multivariable model including sex, "
    "age group, and reporting source. Variance was "
    "clustered at the report level."
)

doc.add_paragraph("")  # linha em branco

# Cria tabela no Word
cols = list(final_table.columns)
table = doc.add_table(rows=1, cols=len(cols))
table.style = "Table Grid"
table.alignment = WD_TABLE_ALIGNMENT.CENTER

# Cabeçalho
hdr_cells = table.rows[0].cells
for j, col in enumerate(cols):
    hdr_cells[j].text = col

# Corpo
for _, row in final_table.iterrows():
    row_cells = table.add_row().cells
    for j, col in enumerate(cols):
        val = row[col]
        row_cells[j].text = "" if (val is None or (isinstance(val, float) and pd.isna(val))) else str(val)

# Formatação básica (fonte)
for r in table.rows:
    for cell in r.cells:
        for p in cell.paragraphs:
            for run in p.runs:
                run.font.size = Pt(9)

doc.save(out_docx)
print(f"✅ Word salvo em: {out_docx}")


Grafico Forest Plot

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION
# ============================================================
FILE_OUT_PNG = (
    "Figure_MedicationError_vs_NoError_Adjusted.png"
)

FILE_OUT_PDF = (
    "Figure_MedicationError_vs_NoError_Adjusted.pdf"
)

PR_COLUMN = "Adj_PR_AnyError"

#FIGURE_TITLE = ()


# ============================================================
# 1. READ RESULTS
# ============================================================

results = final_table.copy()

required_columns = [
    "Variable",
    "Category",
    PR_COLUMN,
]

missing_columns = [
    column
    for column in required_columns
    if column not in results.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns not found: {missing_columns}"
    )


# ============================================================
# 2. PARSE PR AND 95% CI
# ============================================================

def parse_pr_ci(value):

    value = str(value).strip()

    if "Reference" in value or "Ref" in value:
        return 1.0, 1.0, 1.0

    # Accepts either an en dash or a hyphen between CI limits
    match = re.search(
        r"([0-9.]+)\s*\(\s*"
        r"([0-9.]+)\s*[–-]\s*"
        r"([0-9.]+)\s*\)",
        value,
    )

    if match is None:
        return np.nan, np.nan, np.nan

    pr = float(match.group(1))
    lower = float(match.group(2))
    upper = float(match.group(3))

    return pr, lower, upper


parsed = results[PR_COLUMN].apply(parse_pr_ci)

results["PR"] = [
    value[0] for value in parsed
]

results["Lower"] = [
    value[1] for value in parsed
]

results["Upper"] = [
    value[2] for value in parsed
]

results = results.dropna(
    subset=["PR", "Lower", "Upper"]
).copy()


# ============================================================
# 3. Y POSITIONS AND SECTION HEADINGS
# ============================================================

y_positions = []
section_headings = []
section_separators = []

current_y = 0
previous_variable = None

for _, row in results.iterrows():

    variable = row["Variable"]

    if variable != previous_variable:

        if previous_variable is not None:
            section_separators.append(
                current_y - 0.35
            )
            current_y += 0.75

        section_headings.append(
            (variable, current_y)
        )

        current_y += 0.85
        previous_variable = variable

    y_positions.append(current_y)
    current_y += 1

results["Y"] = y_positions


# ============================================================
# 4. FIGURE LAYOUT
# ============================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Arial",
        "DejaVu Sans",
    ],
    "font.size": 12,
})

figure_height = max(
    8,
    current_y * 0.42,
)

fig = plt.figure(
    figsize=(16, figure_height)
)

grid = fig.add_gridspec(
    nrows=1,
    ncols=3,
    width_ratios=[
        0.38,  # category labels
        0.38,  # forest plot
        0.24,  # numerical results
    ],
    wspace=0.04,
)

ax_labels = fig.add_subplot(
    grid[0, 0]
)

ax_plot = fig.add_subplot(
    grid[0, 1]
)

ax_results = fig.add_subplot(
    grid[0, 2],
    sharey=ax_plot,
)


# ============================================================
# 5. COMMON AXIS FORMATTING
# ============================================================

for axis in [
    ax_labels,
    ax_plot,
    ax_results,
]:

    axis.set_ylim(
        current_y + 0.5,
        -0.8,
    )

    axis.get_yaxis().set_visible(False)

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_visible(False)

    for separator in section_separators:
        axis.axhline(
            y=separator,
            color="#D9D9D9",
            linewidth=0.8,
            zorder=0,
        )


# Left labels
ax_labels.get_xaxis().set_visible(False)
ax_labels.spines["bottom"].set_visible(False)
ax_labels.set_xlim(0, 1)

# Right numerical results
ax_results.get_xaxis().set_visible(False)
ax_results.spines["bottom"].set_visible(False)
ax_results.set_xlim(0, 1)


# ============================================================
# 6. VARIABLE HEADINGS AND CATEGORY LABELS
# ============================================================

for variable, heading_y in section_headings:

    ax_labels.text(
        0,
        heading_y,
        variable,
        fontweight="bold",
        fontsize=13,
        va="center",
        ha="left",
    )


for _, row in results.iterrows():

    ax_labels.text(
        0.035,
        row["Y"],
        row["Category"],
        fontsize=12,
        va="center",
        ha="left",
    )


# ============================================================
# 7. FOREST PLOT
# ============================================================

ax_plot.set_xscale("log")

ax_plot.set_xlim(
    0.08,
    6.0,
)

ax_plot.set_xticks([
    0.1,
    0.2,
    0.5,
    1,
    2,
    5,
])

ax_plot.get_xaxis().set_major_formatter(
    plt.ScalarFormatter()
)

ax_plot.grid(
    axis="x",
    linestyle=":",
    color="#D9D9D9",
    linewidth=0.8,
    zorder=0,
)

ax_plot.axvline(
    x=1,
    color="black",
    linestyle="--",
    linewidth=1,
    zorder=1,
)

for _, row in results.iterrows():

    is_reference = (
        "Reference" in str(row[PR_COLUMN])
        or "Ref" in str(row[PR_COLUMN])
    )

    if is_reference:

        ax_plot.scatter(
            row["PR"],
            row["Y"],
            marker="s",
            s=35,
            color="black",
            zorder=3,
        )

    else:

        ax_plot.errorbar(
            x=row["PR"],
            y=row["Y"],
            xerr=[[
                row["PR"] - row["Lower"]
            ], [
                row["Upper"] - row["PR"]
            ]],
            fmt="s",
            markersize=5.5,
            color="black",
            ecolor="black",
            elinewidth=1.3,
            capsize=3.5,
            zorder=3,
        )


# ax_plot.set_title(
#     FIGURE_TITLE,
#     loc="left",
#     fontweight="bold",
#     fontsize=14,
#     pad=14,
# )

ax_plot.set_xlabel(
    "Adjusted PR (95% CI)",
    fontweight="bold",
    fontsize=12,
)


# ============================================================
# 8. NUMERICAL RESULTS
# ============================================================

for _, row in results.iterrows():

    value = str(row[PR_COLUMN])

    if "Reference" in value:
        value = "1.00 (Ref)"

    ax_results.text(
        0,
        row["Y"],
        value,
        fontsize=11,
        va="center",
        ha="left",
    )


# ============================================================
# 9. FOOTNOTE AND SAVE
# ============================================================

# fig.text(
#     0.01,
#     0.008,
#     (
#         "Outcome: any medication error "
#         "(dosing- or non-dosing-related); "
#         "reference outcome group: drug–event pairs "
#         "without medication errors. "
#         "Estimates were adjusted for sex, age group, "
#         "and reporting source. Variance was clustered "
#         "at the report level. Reporting-source "
#         "categories with fewer than 10 drug–event "
#         "pairs were excluded from the regression."
#     ),
#     ha="left",
#     va="bottom",
#     fontsize=9.5,
# )

plt.tight_layout(
    rect=(0, 0.055, 1, 1)
)

fig.savefig(
    FILE_OUT_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FILE_OUT_PDF,
    bbox_inches="tight",
)

print(
    f"Figure saved: {FILE_OUT_PNG}"
)

print(
    f"Figure saved: {FILE_OUT_PDF}"
)

plt.show()